<a href="https://colab.research.google.com/github/amroatefelshafey/ascon/blob/main/ASCON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
# ASCON Implementation

"""
Implementing the ASCON cipher using Python
"""

# ----- Helper Functions taken from the github of ascon library -----
def bytes_to_int(bytes) -> int:
    return int.from_bytes(bytes, 'little')
    return sum([bi << (i*8) for i, bi in enumerate(to_bytes(bytes))])

def int_to_bytes(integer: int, nbytes: int) -> bytes:
    return integer.to_bytes(nbytes, 'little')

def bytes_to_state(bytes: bytes) -> list[int]:
    return [bytes_to_int(bytes[8*w:8*(w+1)]) for w in range(5)]

def printstate(S: list[int], description: str = "") -> None:
    print(" " + description)
    print(" ".join(["{s:016x}".format(s=s) for s in S]))
#------------------------- Functions & Constants -------------------------

IV = 0x00001000808c0001
ROUND_CONSTANTS = [
                   0xf0,0xe1,0xd2,0xc3,
                   0xb4,0xa5,0x96,0x87,
                   0x78,0x69,0x5a,0x4b
                  ]

def zeroOut(num: int) -> bytes:
  """Generate a stream of x00 bytes num times.

     Args:
     num: How many 0 bytes to generate

     Output:
     The zero bytes
  """

  return num * b'\x00'

def rotate(num: int, noOfBits: int, shAmt: int ,right = 1) -> int:
  """ Define a rotate function either right or left (right by default)

    Args:
      num: the number to be rotated
      noOfBits: # of bits in num
      shAmt: How many bits to rotate
      right: Rotate right or left

    Output:
      num: num rotated shAmt times either to the right or the left
  """
  mask = 0x0
  for i in range(noOfBits):
    mask = (mask << 1) | 1

  if (right == 1):
    num = ( ( num << (noOfBits - shAmt)) | (num >> shAmt) ) & mask # Isolate the LSB to the highest-order position such that when we |, we >>>
  else:
    num = ( ( num >> (noOfBits - shAmt)) | (num << shAmt) ) & mask # Isolate the MSB to the lowest-order position such that when we |, we <<<

  return num

#-------------------- ASCON AEAD-128 Building Blocks --------------------

def permutation(State: list[int], rounds: int) -> list[int]:
  """Defines the ASCON permutation function, which is 3 stages as implemented.

     Args:
     State: 320-bit ASCON state
     rounds: # of rounds to permute, either 12 or 8 (12 in init. & finalization,
     8 in intermediate stages like AD processing).

     Output:
     Updated 320-bit state after permuting for a (12) or b (8) rounds.
  """

 #-----round constant addition-----
  for i in range(rounds):
    x = ROUND_CONSTANTS[12 - rounds + i]
    State[2] ^= x

  #-----s-box layer-----
    State[0] ^= State[4];    State[4] ^= State[3];    State[2] ^= State[1];
    t0  = State[0];    t1  = State[1];    t2  = State[2];    t3  = State[3];    t4  = State[4];
    t0 =~ t0;    t1 =~ t1;    t2 =~ t2;    t3 =~ t3;    t4 =~ t4;
    t0 &= State[1];    t1 &= State[2];    t2 &= State[3];    t3 &= State[4];    t4 &= State[0];
    State[0] ^= t1;    State[1] ^= t2;    State[2] ^= t3;    State[3] ^= t4;    State[4] ^= t0;
    State[1] ^= State[0];    State[0] ^= State[4];    State[3] ^= State[2];    State[2] =~ State[2];
  #Inherited from "C instructions" in the ASCON-Specification website
    State[2] &= 0xFFFFFFFFFFFFFFFF #Prevents S[2] from being built in with -ve sign (treating it as signed)

  #-----linear layer-----
    State[0] ^= (rotate(State[0],64,19)) ^ (rotate(State[0],64,28))
    State[1] ^= (rotate(State[1],64,61)) ^ (rotate(State[1],64,39))
    State[2] ^= (rotate(State[2],64,1) ) ^ (rotate(State[2],64,6) )
    State[3] ^= (rotate(State[3],64,10)) ^ (rotate(State[3],64,17))
    State[4] ^= (rotate(State[4],64,7) ) ^ (rotate(State[4],64,41))

  return State


def init(State: list[int], iv: int, key: bytes, nonce: bytes) -> None:
  """ASCON-AEAD128 Initialization function, which forms the state by a simple
     concatenation IV||K||N resulting in 320-bits. A permutation with a=12 rounds
     is run and then the lower two blocks are XORed with the key (128-bit XOR).

     Args:
     State: State construction list of 5 integers
     iv: The IV is a parameter, formula determines it (used as a const here)
     key: the ciphers key, bytes object with length 16
     nonce: the ciphers nonce, bytes object with length 16

     Output:
     None, updates the state
  """
  State[0] = iv
  State[1] = bytes_to_int(key[0:8])     #High Key
  State[2] = bytes_to_int(key[8:])      #Low Key
  State[3] = bytes_to_int(nonce[0:8])   #High Nonce
  State[4] = bytes_to_int(nonce[8:])    #Low Nonce

  State = permutation(State, 12)

  State[3] ^= bytes_to_int(key[0:8])
  State[4] ^= bytes_to_int(key[8:])


def plaintextCipherText(State: list[int], plaintext: bytes) -> bytes:
  """Plaintext to Ciphertext transformation function

     Args:
     State: ASCON 320-bit state
     plaintext: plaintext input (message to be encrypted)

     Output:
     ciphertext: The ciphertext, which has the same length as the plaintext
  """
  remPlaintextLength = len(plaintext) % 16
  padding = 16 - remPlaintextLength
  padding_factor = bytes([0x01]) + zeroOut(padding - 1)

  ciphertext = bytes() #Ciphertext is also bytes just like the plaintext
  for i in range(len(plaintext) // 16):
    State[0] ^=  bytes_to_int(plaintext[16*i:16*i+8])
    State[1] ^=  bytes_to_int(plaintext[16*i+8:16*i+16])
    ciphertext += int_to_bytes(State[0], 8) + int_to_bytes(State[1], 8)
    State = permutation(State, 8)

  padded_plaintext = plaintext + padding_factor
  State[0] ^= bytes_to_int(padded_plaintext[-16:-8])
  State[1] ^= bytes_to_int(padded_plaintext[-8:-1])

  for i in range(remPlaintextLength):
    if (i< 8):
      ciphertext += int_to_bytes(State[0], 8)[i:i+1]
    else:
      ciphertext += int_to_bytes(State[1], 8)[i%8:(i+1)%8]

  return ciphertext


def finalize(State: list[int], key: bytes) -> bytes:
  """ASCON Finalization function

     Args:
     State: ASCON 320-bit state
     key: The 128-bit (16 bytes) key

     Output:
     tag: Authentication tag (128 bits/16 bytes in size) as a bytes object
  """
  tag = bytes()
  State[2] ^= bytes_to_int(key[0:8])
  State[3] ^= bytes_to_int(key[8:])

  State = permutation(State, 12)
  State[3] ^= bytes_to_int(key[0:8])
  State[4] ^= bytes_to_int(key[8:])

  tag = int_to_bytes(State[3], 8) + int_to_bytes(State[4], 8)

  return tag



def encrypt(key: bytes, nonce: bytes, plaintext: bytes) -> bytes:
  """ASCON Encryption function, this functions acts as a top-level,
  utilizing all of the building blocks. The function takes the basic
  parameters and outputs the ciphertext and authentication tag.

  Args:
  key: 16 bytes key
  nonce: 16 bytes nonce
  plaintext: plaintext to be encrypted, any length (bytes object)

  Output:
  {ciphertext, tag} combined in one bytes object.

  """
  S = [0,0,0,0,0] # Initialize a state of 5 integers
  init(S, IV, key, nonce)
  S[4] ^= 1<<63
  ciphertext = plaintextCipherText(S, plaintext)
  tag = finalize(S, key)

  return ciphertext + tag

#--------------------------------------------------------------------

key = b'0123456789ABCDEF'      # 16 bytes
nonce = b'ABCDEF0123456789'    # 16 bytes
ad = b''                       # Associated Data
plaintext = b'Hello World'
encrypt(key, nonce, plaintext).hex()

'368f11357761d55d0dcfab2cc86521a9446ba1acc3b9dd0b4b50f9'

In [ ]:
# @title
# Rotate function, Zeroes Unit Testing
import random


"""
def rotate(num:bitarray,noOfBits,shAmt,right = 1):
  mask = 0x0
  for i in range(noOfBits):
    mask = (mask << 1) | 1
  specifier = f"0{noOfBits}b"

  for i in range(shAmt):
    if (right == 1):
      num = (ba2int(num) << noOfBits - 1) & mask | (ba2int(num) >> 1)
      num = format(num, specifier) # Must be a string in order to convert to bitarray
      num = bitarray(num)
    else:
      num = (ba2int(num) >> noOfBits - 1) & mask | (ba2int(num) << 1)
      num = format(num, specifier)
      num = bitarray(num)

  return num
"""

"""
def rotate(num:bitarray,noOfBits,shAmt,right = 1):
  mask = 0x0
  for i in range(noOfBits):
    mask = (mask << 1) | 1
  specifier = f"0{noOfBits}b"


  if (right == 1):
    num = ( (ba2int(num) << (noOfBits - shAmt)) | (ba2int(num) >> shAmt) ) & mask # Isolate the LSB to the highest-order position such that when we |, we >>>
  else:
    num = ( (ba2int(num) >> (noOfBits - shAmt)) | (ba2int(num) << shAmt) ) & mask # Isolate the MSB to the lowest-order position such that when we |, we <<<

  num = format(num, specifier) # Must be a string in order to convert to bitarray
  return bitarray(num)
"""


def rotate(num:int,noOfBits,shAmt,right = 1):
  mask = 0x0
  for i in range(noOfBits):
    mask = (mask << 1) | 1


  if (right == 1):
    num = ( ( num << (noOfBits - shAmt)) | (num >> shAmt) ) & mask # Isolate the LSB to the highest-order position such that when we |, we >>>
  else:
    num = ( ( num >> (noOfBits - shAmt)) | (num << shAmt) ) & mask # Isolate the MSB to the lowest-order position such that when we |, we <<<

  return num


def zeroOut(num:int) -> bytes:
  """Generate a stream of x00 bytes num times.

     Args:
     num: How many 0 bytes to generate

     Output:
     The zero bytes
  """

  return num * b'\x00'

for i in range(0,5):
  int_value = random.randint(0,0xFFFF)
  print("\n",hex(int_value))

  print(f"The number rotated {i} times is",hex(rotate(int_value,16,i,1)),end="\n")

zeroOut(5)


 0xe617
The number rotated 0 times is 0xe617

 0x4a2e
The number rotated 1 times is 0x2517

 0x2e72
The number rotated 2 times is 0x8b9c

 0x9172
The number rotated 3 times is 0x522e

 0x1c1a
The number rotated 4 times is 0xa1c1


b'\x00\x00\x00\x00\x00'

In [ ]:
# @title
#Permutation Function Unit Test


# Dependencies
def printwords(S: list[int], description: str = "") -> None:
    print(" " + description)
    print("\n".join(["  x{i}={s:016x}".format(**locals()) for i, s in enumerate(S)]))

def rotr(val: int, r: int) -> int:
    return (val >> r) | ((val & (1<<r)-1) << (64-r))

debugpermutation = True
#-----------------------------------------------------------------------------

ROUND_CONSTANTS = [
                   0xf0,0xe1,0xd2,0xc3,
                   0xb4,0xa5,0x96,0x87,
                   0x78,0x69,0x5a,0x4b
                  ]

def rotate(num:int, noOfBits:int, shAmt:int ,right = 1):
  mask = 0x0
  for i in range(noOfBits):
    mask = (mask << 1) | 1

  if (right == 1):
    num = ( ( num << (noOfBits - shAmt)) | (num >> shAmt) ) & mask # Isolate the LSB to the highest-order position such that when we |, we >>>
  else:
    num = ( ( num >> (noOfBits - shAmt)) | (num << shAmt) ) & mask # Isolate the MSB to the lowest-order position such that when we |, we <<<

  return num

#-----------------------------ascon library permutation function-------------------------------------

def ascon_permutation(S: list[int], rounds: int=1):
    """
    Ascon core permutation for the sponge construction - internal helper function.
    S: Ascon state, a list of 5 64-bit integers
    rounds: number of rounds to perform
    returns nothing, updates S
    """
    assert rounds <= 12
    if debugpermutation: printwords(S, "permutation input:")
    for r in range(12-rounds, 12):
        # --- add round constants ---
        S[2] ^= (0xf0 - r*0x10 + r*0x1)
        if debugpermutation: printwords(S, "round constant addition:")
        # --- substitution layer ---
        S[0] ^= S[4]
        S[4] ^= S[3]
        S[2] ^= S[1]
        T = [(S[i] ^ 0xFFFFFFFFFFFFFFFF) & S[(i+1)%5] for i in range(5)]
        for i in range(5):
            S[i] ^= T[(i+1)%5]
        S[1] ^= S[0]
        S[0] ^= S[4]
        S[3] ^= S[2]
        S[2] ^= 0XFFFFFFFFFFFFFFFF
        if debugpermutation: printwords(S, "substitution layer:")
        # --- linear diffusion layer ---
        S[0] ^= rotr(S[0], 19) ^ rotr(S[0], 28)
        S[1] ^= rotr(S[1], 61) ^ rotr(S[1], 39)
        S[2] ^= rotr(S[2],  1) ^ rotr(S[2],  6)
        S[3] ^= rotr(S[3], 10) ^ rotr(S[3], 17)
        S[4] ^= rotr(S[4],  7) ^ rotr(S[4], 41)
        if debugpermutation: printwords(S, "linear diffusion layer:")

#---------------------------Unit Under Test (UUT)-------------------------------------

def permutation(State:list[int], rounds:int):

  for i in range(rounds):
    x = ROUND_CONSTANTS[12 - rounds + i]
    State[2] ^= x

  #s-box layer
    State[0] ^= State[4];    State[4] ^= State[3];    State[2] ^= State[1];
    t0  = State[0];    t1  = State[1];    t2  = State[2];    t3  = State[3];    t4  = State[4];
    t0 =~ t0;    t1 =~ t1;    t2 =~ t2;    t3 =~ t3;    t4 =~ t4;
    t0 &= State[1];    t1 &= State[2];    t2 &= State[3];    t3 &= State[4];    t4 &= State[0];
    State[0] ^= t1;    State[1] ^= t2;    State[2] ^= t3;    State[3] ^= t4;    State[4] ^= t0;
    State[1] ^= State[0];    State[0] ^= State[4];    State[3] ^= State[2];    State[2] =~ State[2];
  #Inherited from "C instructions" in the ASCON-Specification website
    State[2] &= 0xFFFFFFFFFFFFFFFF #Prevents S[2] from being built in with -ve sign (treating it as signed)

    State[0] ^= (rotate(State[0],64,19)) ^ (rotate(State[0],64,28))
    State[1] ^= (rotate(State[1],64,61)) ^ (rotate(State[1],64,39))
    State[2] ^= (rotate(State[2],64,1) ) ^ (rotate(State[2],64,6) )
    State[3] ^= (rotate(State[3],64,10)) ^ (rotate(State[3],64,17))
    State[4] ^= (rotate(State[4],64,7) ) ^ (rotate(State[4],64,41))

  return State


ascon_permutation([192,4039,4489,3543,657],1)
printwords(permutation([192,4039,4489,3543,657],1))



 permutation input:
  x0=00000000000000c0
  x1=0000000000000fc7
  x2=0000000000001189
  x3=0000000000000dd7
  x4=0000000000000291
 round constant addition:
  x0=00000000000000c0
  x1=0000000000000fc7
  x2=00000000000011c2
  x3=0000000000000dd7
  x4=0000000000000291
 substitution layer:
  x0=0000000000001091
  x1=0000000000001c44
  x2=ffffffffffffe3fa
  x3=00000000000011c3
  x4=00000000000002c0
 linear diffusion layer:
  x0=0213291000001091
  x1=000000388800fe64
  x2=6bffffffffffed88
  x3=78218000000011c7
  x4=80000001600002c5
 
  x0=0213291000001091
  x1=000000388800fe64
  x2=6bffffffffffed88
  x3=78218000000011c7
  x4=80000001600002c5


In [ ]:
# @title
#Initilization Unit Testing

# Dependencies
def to_bytes(l) -> bytes:
    # where l is a list or bytearray or bytes
    return bytes(l)
def zero_bytes(n: int) -> bytes:
    return n * b"\x00"

def int_to_bytes(integer: int, nbytes: int) -> bytes:
    return integer.to_bytes(nbytes, 'little')

def bytes_to_int(bytes) -> int:
    return int.from_bytes(bytes, 'little')
    return sum([bi << (i*8) for i, bi in enumerate(to_bytes(bytes))])

def bytes_to_state(bytes: bytes) -> list[int]:
    return [bytes_to_int(bytes[8*w:8*(w+1)]) for w in range(5)]

def printstate(S: list[int], description: str = "") -> None:
    print(" " + description)
    print(" ".join(["{s:016x}".format(s=s) for s in S]))

def rotr(val: int, r: int) -> int:
    return (val >> r) | ((val & (1<<r)-1) << (64-r))

def ascon_permutation(S: list[int], rounds: int=1):
    """
    Ascon core permutation for the sponge construction - internal helper function.
    S: Ascon state, a list of 5 64-bit integers
    rounds: number of rounds to perform
    returns nothing, updates S
    """
    assert rounds <= 12
    if debugpermutation: printwords(S, "permutation input:")
    for r in range(12-rounds, 12):
        # --- add round constants ---
        S[2] ^= (0xf0 - r*0x10 + r*0x1)
        if debugpermutation: printwords(S, "round constant addition:")
        # --- substitution layer ---
        S[0] ^= S[4]
        S[4] ^= S[3]
        S[2] ^= S[1]
        T = [(S[i] ^ 0xFFFFFFFFFFFFFFFF) & S[(i+1)%5] for i in range(5)]
        for i in range(5):
            S[i] ^= T[(i+1)%5]
        S[1] ^= S[0]
        S[0] ^= S[4]
        S[3] ^= S[2]
        S[2] ^= 0XFFFFFFFFFFFFFFFF
        if debugpermutation: printwords(S, "substitution layer:")
        # --- linear diffusion layer ---
        S[0] ^= rotr(S[0], 19) ^ rotr(S[0], 28)
        S[1] ^= rotr(S[1], 61) ^ rotr(S[1], 39)
        S[2] ^= rotr(S[2],  1) ^ rotr(S[2],  6)
        S[3] ^= rotr(S[3], 10) ^ rotr(S[3], 17)
        S[4] ^= rotr(S[4],  7) ^ rotr(S[4], 41)
        if debugpermutation: printwords(S, "linear diffusion layer:")

debug = True
debugpermutation = False

#----------------------------------------------------------------------------

ROUND_CONSTANTS = [
                   0xf0,0xe1,0xd2,0xc3,
                   0xb4,0xa5,0x96,0x87,
                   0x78,0x69,0x5a,0x4b
                  ]
IV = 0x00001000808c0001
#IV(not little-endian) = 0x01008c8000100000
#IV(old) = 0x0100c80080100000

def rotate(num:int, noOfBits:int, shAmt:int ,right = 1):
  mask = 0x0
  for i in range(noOfBits):
    mask = (mask << 1) | 1

  if (right == 1):
    num = ( ( num << (noOfBits - shAmt)) | (num >> shAmt) ) & mask # Isolate the LSB to the highest-order position such that when we |, we >>>
  else:
    num = ( ( num >> (noOfBits - shAmt)) | (num << shAmt) ) & mask # Isolate the MSB to the lowest-order position such that when we |, we <<<

  return num

def permutation(State:list[int], rounds:int):

  for i in range(rounds):
    x = ROUND_CONSTANTS[12 - rounds + i]
    State[2] ^= x

  #s-box layer
    State[0] ^= State[4];    State[4] ^= State[3];    State[2] ^= State[1];
    t0  = State[0];    t1  = State[1];    t2  = State[2];    t3  = State[3];    t4  = State[4];
    t0 =~ t0;    t1 =~ t1;    t2 =~ t2;    t3 =~ t3;    t4 =~ t4;
    t0 &= State[1];    t1 &= State[2];    t2 &= State[3];    t3 &= State[4];    t4 &= State[0];
    State[0] ^= t1;    State[1] ^= t2;    State[2] ^= t3;    State[3] ^= t4;    State[4] ^= t0;
    State[1] ^= State[0];    State[0] ^= State[4];    State[3] ^= State[2];    State[2] =~ State[2];
  #Inherited from "C instructions" in the ASCON-Specification website
    State[2] &= 0xFFFFFFFFFFFFFFFF #Prevents S[2] from being built in with -ve sign (treating it as signed)

    State[0] ^= (rotate(State[0],64,19)) ^ (rotate(State[0],64,28))
    State[1] ^= (rotate(State[1],64,61)) ^ (rotate(State[1],64,39))
    State[2] ^= (rotate(State[2],64,1) ) ^ (rotate(State[2],64,6) )
    State[3] ^= (rotate(State[3],64,10)) ^ (rotate(State[3],64,17))
    State[4] ^= (rotate(State[4],64,7) ) ^ (rotate(State[4],64,41))

  return State
#-------------------------ascon library initilization function-----------------------

def ascon_initialize(S: list[int], k: int, rate: int, a: int, b: int, version: int, key: bytes, nonce: bytes):
    """
    Ascon initialization phase - internal helper function.
    S: Ascon state, a list of 5 64-bit integers
    k: key size in bits
    rate: block size in bytes (16 for Ascon-AEAD128)
    a: number of initialization/finalization rounds for permutation
    b: number of intermediate rounds for permutation
    version: 1 (for Ascon-AEAD128)
    key: a bytes object of size 16 (for Ascon-AEAD128; 128-bit security)
    nonce: a bytes object of size 16
    returns nothing, updates S
    """
    taglen = 128
    iv = to_bytes([version, 0, (b<<4) + a]) + int_to_bytes(taglen, 2) + to_bytes([rate, 0, 0])
    if debug: print("IV:", iv.hex()) #I added this to monitor the value of IV
    S[0], S[1], S[2], S[3], S[4] = bytes_to_state(iv + key + nonce)
    if debug: printstate(S, "initial value:")

    ascon_permutation(S, a)

    zero_key = bytes_to_state(zero_bytes(40-len(key)) + key)
    S[0] ^= zero_key[0]
    S[1] ^= zero_key[1]
    S[2] ^= zero_key[2]
    S[3] ^= zero_key[3]
    S[4] ^= zero_key[4]
    if debug: printstate(S, "initialization:")

#--------------------------Unit Under Test (UUT)-------------------------------

def init(State:list[int], iv:int, key:bytes, nonce:bytes):
  State[0] = iv
  State[1] = bytes_to_int(key[0:8])     #High Key
  State[2] = bytes_to_int(key[8:])      #Low Key
  State[3] = bytes_to_int(nonce[0:8])   #High Nonce
  State[4] = bytes_to_int(nonce[8:])    #Low Nonce
  if debug: printstate(State,"\nUUT initial value") #
  State = permutation(State, 12)
  State[3] ^= bytes_to_int(key[0:8]) #
  State[4] ^= bytes_to_int(key[8:])  #

  return State

key = b'0123456789ABCDEF'
nonce = b'0123456789ABCDEF'

ascon_initialize([1,2,3,4,5], 128, 16, 12, 8, 1, key, nonce)
printstate(init([1,2,3,4,5],IV,key,nonce))


"""Summary of changes:
   1 - Fixed an error of State[3:] being XORed ^= with key. Cant do that since key is a bytes and State[3:] was list[int].
   Moreover, State[3:] ^= bytes_to_int(key) still didnt work as well because ^ between list[int] and int is not legal.
   2 - Added a debug signal to print the state before permuting it and after its first initalization. This helped in comparison
   with the the library's state after initialization and in understanding the way the IV is assigned which is confusing due to
   its endianness
"""

IV: 01008c8000100000
 initial value:
00001000808c0001 3736353433323130 4645444342413938 3736353433323130 4645444342413938
 initialization:
7ea3d00493b77c43 1c94fcbcd1545fc4 ee0d2c6d7684f200 05d42bb495958890 598f8f94111e0315
 
UUT initial value
00001000808c0001 3736353433323130 4645444342413938 3736353433323130 4645444342413938
 
7ea3d00493b77c43 1c94fcbcd1545fc4 ee0d2c6d7684f200 05d42bb495958890 598f8f94111e0315


In [ ]:
# @title
#Plaintext & Finalize Integration Test

# Dependencies
def zero_bytes(n: int) -> bytes:
    return n * b"\x00"

def to_bytes(l) -> bytes:
    # where l is a list or bytearray or bytes
    return bytes(l)

def bytes_to_int(bytes) -> int:
    return int.from_bytes(bytes, 'little')
    return sum([bi << (i*8) for i, bi in enumerate(to_bytes(bytes))])

def int_to_bytes(integer: int, nbytes: int) -> bytes:
    return integer.to_bytes(nbytes, 'little')

def printstate(S: list[int], description: str = "") -> None:
    print(" " + description)
    print(" ".join(["{s:016x}".format(s=s) for s in S]))

def rotr(val: int, r: int) -> int:
    return (val >> r) | ((val & (1<<r)-1) << (64-r))

def ascon_permutation(S: list[int], rounds: int=1):
    """
    Ascon core permutation for the sponge construction - internal helper function.
    S: Ascon state, a list of 5 64-bit integers
    rounds: number of rounds to perform
    returns nothing, updates S
    """
    assert rounds <= 12
    if debugpermutation: printwords(S, "permutation input:")
    for r in range(12-rounds, 12):
        # --- add round constants ---
        S[2] ^= (0xf0 - r*0x10 + r*0x1)
        if debugpermutation: printwords(S, "round constant addition:")
        # --- substitution layer ---
        S[0] ^= S[4]
        S[4] ^= S[3]
        S[2] ^= S[1]
        T = [(S[i] ^ 0xFFFFFFFFFFFFFFFF) & S[(i+1)%5] for i in range(5)]
        for i in range(5):
            S[i] ^= T[(i+1)%5]
        S[1] ^= S[0]
        S[0] ^= S[4]
        S[3] ^= S[2]
        S[2] ^= 0XFFFFFFFFFFFFFFFF
        if debugpermutation: printwords(S, "substitution layer:")
        # --- linear diffusion layer ---
        S[0] ^= rotr(S[0], 19) ^ rotr(S[0], 28)
        S[1] ^= rotr(S[1], 61) ^ rotr(S[1], 39)
        S[2] ^= rotr(S[2],  1) ^ rotr(S[2],  6)
        S[3] ^= rotr(S[3], 10) ^ rotr(S[3], 17)
        S[4] ^= rotr(S[4],  7) ^ rotr(S[4], 41)
        if debugpermutation: printwords(S, "linear diffusion layer:")

debug = True
debugpermutation = False

#----------------------------------------------------------------------------

ROUND_CONSTANTS = [
                   0xf0,0xe1,0xd2,0xc3,
                   0xb4,0xa5,0x96,0x87,
                   0x78,0x69,0x5a,0x4b
                  ]

def rotate(num:int, noOfBits:int, shAmt:int ,right = 1):
  mask = 0x0
  for i in range(noOfBits):
    mask = (mask << 1) | 1

  if (right == 1):
    num = ( ( num << (noOfBits - shAmt)) | (num >> shAmt) ) & mask # Isolate the LSB to the highest-order position such that when we |, we >>>
  else:
    num = ( ( num >> (noOfBits - shAmt)) | (num << shAmt) ) & mask # Isolate the MSB to the lowest-order position such that when we |, we <<<

  return num

def zeroOut(num:int) -> bytes:
  """Generate a stream of x00 bytes num times.

     Args:
     num: How many 0 bytes to generate

     Output:
     The zero bytes
  """

  return num * b'\x00'

def permutation(State:list[int], rounds:int):

  for i in range(rounds):
    x = ROUND_CONSTANTS[12 - rounds + i]
    State[2] ^= x

  #s-box layer
    State[0] ^= State[4];    State[4] ^= State[3];    State[2] ^= State[1];
    t0  = State[0];    t1  = State[1];    t2  = State[2];    t3  = State[3];    t4  = State[4];
    t0 =~ t0;    t1 =~ t1;    t2 =~ t2;    t3 =~ t3;    t4 =~ t4;
    t0 &= State[1];    t1 &= State[2];    t2 &= State[3];    t3 &= State[4];    t4 &= State[0];
    State[0] ^= t1;    State[1] ^= t2;    State[2] ^= t3;    State[3] ^= t4;    State[4] ^= t0;
    State[1] ^= State[0];    State[0] ^= State[4];    State[3] ^= State[2];    State[2] =~ State[2];
  #Inherited from "C instructions" in the ASCON-Specification website
    State[2] &= 0xFFFFFFFFFFFFFFFF #Prevents S[2] from being built in with -ve sign (treating it as signed)

    State[0] ^= (rotate(State[0],64,19)) ^ (rotate(State[0],64,28))
    State[1] ^= (rotate(State[1],64,61)) ^ (rotate(State[1],64,39))
    State[2] ^= (rotate(State[2],64,1) ) ^ (rotate(State[2],64,6) )
    State[3] ^= (rotate(State[3],64,10)) ^ (rotate(State[3],64,17))
    State[4] ^= (rotate(State[4],64,7) ) ^ (rotate(State[4],64,41))

  return State
#-------------------------ascon library process plaintext function & finalize function-----------------------

def ascon_process_plaintext(S: list[int], b: int, rate: int, plaintext):
    """
    Ascon plaintext processing phase (during encryption) - internal helper function.
    S: Ascon state, a list of 5 64-bit integers
    b: number of intermediate rounds for permutation
    rate: block size in bytes (16 for Ascon-AEAD128)
    plaintext: a bytes object of arbitrary length
    returns the ciphertext (without tag), updates S
    """
    p_lastlen = len(plaintext) % rate
    p_padding = to_bytes([0x01]) + zero_bytes(rate-p_lastlen-1)
    p_padded = to_bytes(plaintext) + p_padding

    # first t-1 blocks
    ciphertext = to_bytes([])
    for block in range(0, len(p_padded) - rate, rate):
        S[0] ^= bytes_to_int(p_padded[block:block+8])
        S[1] ^= bytes_to_int(p_padded[block+8:block+16])
        ciphertext += (int_to_bytes(S[0], 8) + int_to_bytes(S[1], 8))
        ascon_permutation(S, b)

    # last block t
    block = len(p_padded) - rate
    S[0] ^= bytes_to_int(p_padded[block:block+8])
    S[1] ^= bytes_to_int(p_padded[block+8:block+16])
    ciphertext += (int_to_bytes(S[0], 8)[:min(8,p_lastlen)] + int_to_bytes(S[1], 8)[:max(0,p_lastlen-8)])
    if debug: printstate(S, "process plaintext:")
    return ciphertext

def ascon_finalize(S: list[int], rate: int, a: int, key):
    """
    Ascon finalization phase - internal helper function.
    S: Ascon state, a list of 5 64-bit integers
    rate: block size in bytes (16 for Ascon-AEAD128)
    a: number of initialization/finalization rounds for permutation
    key: a bytes object of size 16 (for Ascon-AEAD128; 128-bit security)
    returns the tag, updates S
    """
    assert len(key) == 16
    S[rate//8+0] ^= bytes_to_int(key[0:8])
    S[rate//8+1] ^= bytes_to_int(key[8:16])

    ascon_permutation(S, a)

    S[3] ^= bytes_to_int(key[-16:-8])
    S[4] ^= bytes_to_int(key[-8:])
    tag = int_to_bytes(S[3], 8) + int_to_bytes(S[4], 8)
    if debug: printstate(S, "finalization:")
    return tag
#--------------------------Unit Under Test (UUT)-------------------------------

def plaintextCipherText(State:list[int], plaintext:bytes) -> bytes:
  remPlaintextLength = len(plaintext) % 16
  padding = 16 - remPlaintextLength
  padding_factor = bytes([0x01]) + zeroOut(padding - 1)

  ciphertext = bytes() #Ciphertext is also bytes just like the plaintext
  for i in range(len(plaintext) // 16):
    State[0] ^=  bytes_to_int(plaintext[16*i:16*i+8])
    State[1] ^=  bytes_to_int(plaintext[16*i+8:16*i+16])
    ciphertext += int_to_bytes(State[0], 8) + int_to_bytes(State[1], 8)
    State = permutation(State, 8)

  padded_plaintext = plaintext + padding_factor
  State[0] ^= bytes_to_int(padded_plaintext[-16:-8])
  State[1] ^= bytes_to_int(padded_plaintext[-8:-1])

  for i in range(remPlaintextLength):
    if (i< 8):
      ciphertext += int_to_bytes(State[0], 8)[i:i+1] #
    else:
      ciphertext += int_to_bytes(State[1], 8)[i%8:(i+1)%8] #

  return ciphertext #


def finalize(State, key) -> bytes:
  tag = bytes()
  State[2] ^= bytes_to_int(key[0:8])
  State[3] ^= bytes_to_int(key[8:])

  State = permutation(State, 12)
  State[3] ^= bytes_to_int(key[0:8])
  State[4] ^= bytes_to_int(key[8:])

  tag = int_to_bytes(State[3], 8) + int_to_bytes(State[4], 8)
  if debug: printstate(State, "uut finalization:") #Added here for testing

  return tag




state = [0x7ea3d00493b77c43, 0x1c94fcbcd1545fc4, 0xee0d2c6d7684f200, 0x05d42bb495958890, 0x598f8f94111e0315]
uut_state = [0x7ea3d00493b77c43, 0x1c94fcbcd1545fc4, 0xee0d2c6d7684f200, 0x05d42bb495958890, 0x598f8f94111e0315]
key = b'HellothereworlCD'
plaintext = b'bytebytebytebytebytebytebytebyte6'

ciphertext = ascon_process_plaintext(state, 8, 16, plaintext)
print(ciphertext)
print("-----------------------------")
uut_ciphertext = plaintextCipherText(uut_state, plaintext)
printstate(uut_state)
print(uut_ciphertext)

tag = ascon_finalize(state, 16, 12, key)
uut_tag = finalize(uut_state, key)

print(tag)
print(uut_tag)

"""Summary of Changes:
1 - Error in the last loop, the [i:i+1] & [i%8:(i+1)%8] were not included,
indexing out a byte from a bytes object returns an int (ASCII interpretation if
its a letter). This was resolved by indexing out a byte as a native bytes object
by including a 1 byte slice using [i:i+1] & [i%8:(i+1)%8]. The %8 in the State[1]
is to wrap around the value of i to 0 as it only runs when i >= 8.
2 - The finalize function was initially planned to be tested independently in its
own cell, but it was was not worth wasting time to copy paste all contents and
helper functions etc. again. Including it here allowed for integration testing
of the plaintext and finalization functions together, which showed me a problem
I spent way too much time trying to solve. Despite both codes looking identical except
that the outputs were different, I ran it in another cell but it was correct! The
problem lied in the fact that the plaintext function I wrote return both the state
and the ciphertext when it should only return the ciphertext, I do not know exactly why
this caused a problem but it may be due to reassignment because the uut_state is already a
parameter in the plaintext function. Anyways removing it fixed the issue and now both the
finalized states and consequently, the tags, match each other and the functionality is
verified.
"""

 process plaintext:
ada934d7a77e876e 77878e6662a4cdd8 90cb5076b7fae22b 6e02899b6fd854f7 4731603fb1ebff58
b'!\x05\xc3\xf6f\xa9\xd7\x1b\xa6& \xb4\xde\x85\xe0y0\x1c\x05\xbe\xb2\xe4H\xa2\xe0J\xb7\x9d}\t\x9e\xabn'
-----------------------------
 
ada934d7a77e876e 77878e6662a4cdd8 90cb5076b7fae22b 6e02899b6fd854f7 4731603fb1ebff58
b'!\x05\xc3\xf6f\xa9\xd7\x1b\xa6& \xb4\xde\x85\xe0y0\x1c\x05\xbe\xb2\xe4H\xa2\xe0J\xb7\x9d}\t\x9e\xabn'
 finalization:
c307fcd0afa466ce 609c4c735bd13bc4 fa58bb02d8868dc5 126437d362c3f9c3 2f9d6e4bd9c8a01a
 uut finalization:
c307fcd0afa466ce 609c4c735bd13bc4 fa58bb02d8868dc5 126437d362c3f9c3 2f9d6e4bd9c8a01a
b'\xc3\xf9\xc3b\xd37d\x12\x1a\xa0\xc8\xd9Kn\x9d/'
b'\xc3\xf9\xc3b\xd37d\x12\x1a\xa0\xc8\xd9Kn\x9d/'


In [ ]:
# @title
#System Testing
import ascon

# Ascon-128 requires:
# Key: 16 bytes (128 bits)
# Nonce: 16 bytes (128 bits)
# Associated Data: bytes (can be empty b'')
# Plaintext: bytes

# ----- Helper Functions taken from the github of ascon library -----
def bytes_to_int(bytes) -> int:
    return int.from_bytes(bytes, 'little')
    return sum([bi << (i*8) for i, bi in enumerate(to_bytes(bytes))])

def int_to_bytes(integer: int, nbytes: int) -> bytes:
    return integer.to_bytes(nbytes, 'little')

def bytes_to_state(bytes: bytes) -> list[int]:
    return [bytes_to_int(bytes[8*w:8*(w+1)]) for w in range(5)]

def printstate(S: list[int], description: str = "") -> None:
    print(" " + description)
    print(" ".join(["{s:016x}".format(s=s) for s in S]))
#--------------------------------------------------------------------

IV = 0x00001000808c0001
ROUND_CONSTANTS = [
                   0xf0,0xe1,0xd2,0xc3,
                   0xb4,0xa5,0x96,0x87,
                   0x78,0x69,0x5a,0x4b
                  ]

def zeroOut(num: int) -> bytes:

  return num * b'\x00'

def rotate(num: int, noOfBits: int, shAmt: int ,right = 1) -> int:

  mask = 0x0
  for i in range(noOfBits):
    mask = (mask << 1) | 1

  if (right == 1):
    num = ( ( num << (noOfBits - shAmt)) | (num >> shAmt) ) & mask # Isolate the LSB to the highest-order position such that when we |, we >>>
  else:
    num = ( ( num >> (noOfBits - shAmt)) | (num << shAmt) ) & mask # Isolate the MSB to the lowest-order position such that when we |, we <<<

  return num

#--------------------------------------------------------------------

def permutation(State: list[int], rounds: int) -> list[int]:

 #-----round constant addition-----
  for i in range(rounds):
    x = ROUND_CONSTANTS[12 - rounds + i]
    State[2] ^= x

  #-----s-box layer-----
    State[0] ^= State[4];    State[4] ^= State[3];    State[2] ^= State[1];
    t0  = State[0];    t1  = State[1];    t2  = State[2];    t3  = State[3];    t4  = State[4];
    t0 =~ t0;    t1 =~ t1;    t2 =~ t2;    t3 =~ t3;    t4 =~ t4;
    t0 &= State[1];    t1 &= State[2];    t2 &= State[3];    t3 &= State[4];    t4 &= State[0];
    State[0] ^= t1;    State[1] ^= t2;    State[2] ^= t3;    State[3] ^= t4;    State[4] ^= t0;
    State[1] ^= State[0];    State[0] ^= State[4];    State[3] ^= State[2];    State[2] =~ State[2];
  #Inherited from "C instructions" in the ASCON-Specification website
    State[2] &= 0xFFFFFFFFFFFFFFFF #Prevents S[2] from being built in with -ve sign (treating it as signed)

  #-----linear layer-----
    State[0] ^= (rotate(State[0],64,19)) ^ (rotate(State[0],64,28))
    State[1] ^= (rotate(State[1],64,61)) ^ (rotate(State[1],64,39))
    State[2] ^= (rotate(State[2],64,1) ) ^ (rotate(State[2],64,6) )
    State[3] ^= (rotate(State[3],64,10)) ^ (rotate(State[3],64,17))
    State[4] ^= (rotate(State[4],64,7) ) ^ (rotate(State[4],64,41))

  return State


def init(State: list[int], iv: int, key: bytes, nonce: bytes) -> None:

  State[0] = iv
  State[1] = bytes_to_int(key[0:8])     #High Key
  State[2] = bytes_to_int(key[8:])      #Low Key
  State[3] = bytes_to_int(nonce[0:8])   #High Nonce
  State[4] = bytes_to_int(nonce[8:])    #Low Nonce

  State = permutation(State, 12)

  State[3] ^= bytes_to_int(key[0:8])
  State[4] ^= bytes_to_int(key[8:])


def plaintextCipherText(State: list[int], plaintext: bytes) -> bytes:

  remPlaintextLength = len(plaintext) % 16
  padding = 16 - remPlaintextLength
  padding_factor = bytes([0x01]) + zeroOut(padding - 1)

  ciphertext = bytes() #Ciphertext is also bytes just like the plaintext
  for i in range(len(plaintext) // 16):
    State[0] ^=  bytes_to_int(plaintext[16*i:16*i+8])
    State[1] ^=  bytes_to_int(plaintext[16*i+8:16*i+16])
    ciphertext += int_to_bytes(State[0], 8) + int_to_bytes(State[1], 8)
    State = permutation(State, 8)

  padded_plaintext = plaintext + padding_factor
  State[0] ^= bytes_to_int(padded_plaintext[-16:-8])
  State[1] ^= bytes_to_int(padded_plaintext[-8:-1])

  for i in range(remPlaintextLength):
    if (i< 8):
      ciphertext += int_to_bytes(State[0], 8)[i:i+1]
    else:
      ciphertext += int_to_bytes(State[1], 8)[i%8:(i+1)%8]

  return ciphertext


def finalize(State: list[int], key: bytes) -> bytes:

  tag = bytes()
  State[2] ^= bytes_to_int(key[0:8])
  State[3] ^= bytes_to_int(key[8:])

  State = permutation(State, 12)
  State[3] ^= bytes_to_int(key[0:8])
  State[4] ^= bytes_to_int(key[8:])

  tag = int_to_bytes(State[3], 8) + int_to_bytes(State[4], 8)

  return tag



def encrypt(key: bytes, nonce: bytes, plaintext: bytes) -> bytes:
  S = [0,0,0,0,0]
  init(S, IV, key, nonce)
  S[4] ^= 1<<63
  ciphertext = plaintextCipherText(S, plaintext)
  tag = finalize(S, key)

  return ciphertext + tag

#--------------------------------------------------------------------

ascon.debug = True
ascon.debugpermutation = True
key = b'0123456789ABCDEF'      # 16 bytes
nonce = b'ABCDEF0123456789'    # 16 bytes
ad = b''                       # Associated Data
plaintext = b'ascon'
uut_result = encrypt(key, nonce, plaintext)

# The correct function in the 'ascon' library is:
variant = 'Ascon-128'
result = ascon_encrypt(key, nonce, ad, plaintext)

print(f"Result: {result.hex()}")
print(f"UUT_Result: {uut_result.hex()}")

Result: 1f991e3676278e5d49affeac1e4f1a4cf7942c73da
UUT_Result: 1f991e3676278e5d49affeac1e4f1a4cf7942c73da


In [ ]:
# @title
#General Testing

#This serves as a general testing cell for learning more about the helper functions
#as well as the bytes object functionality. Small 'play tests' allow me to gain
#a better understanding of how these functions work.

def bytes_to_int(bytes) -> int:
    return int.from_bytes(bytes, 'little')
    return sum([bi << (i*8) for i, bi in enumerate(to_bytes(bytes))])

def bytes_to_state(bytes: bytes) -> list[int]:
    return [bytes_to_int(bytes[8*w:8*(w+1)]) for w in range(5)]

def zero_bytes(n: int) -> bytes:
    return n * b"\x00"

def to_bytes(l) -> bytes:
    # where l is a list or bytearray or bytes
    return bytes(l)




iv = bytes(5) # 5 bytes initilization
iv = b'0101'
iv[0] #ASCII prints 48 as that is 0 in ASCII (int)
type(iv) #bytes
iv[0:1]

key=b'1001'
nonce=b'0011'

m = iv+key+nonce # + Actually concatenates!
list(m) # List with each byte element in ASCII

iv=0x1092
type(iv) # int

plaintext = b'0123456789ABCDEF'
rate = 16
p_lastlen = len(plaintext) % rate
p_padding = to_bytes([0x01]) + zero_bytes(rate-p_lastlen-1)
p_padded = to_bytes(plaintext) + p_padding
p_padded
to_bytes([0x01])
a = bytes()
a += b'1101' + b'12'
a

b'110112'

In [ ]:
# @title
%pip install ascon

In [ ]:
# @title
#!/usr/bin/env python3
"""
Implementation of Ascon, an authenticated cipher and hash function
NIST SP 800-232
https://ascon.iaik.tugraz.at/
"""
from __future__ import annotations

from typing import Literal, TypeAlias, Iterable

BytesLike: TypeAlias = bytes|bytearray|memoryview

AsconAeadVariant: TypeAlias = Literal[
    "Ascon-AEAD128",
]

AsconHashVariant: TypeAlias = Literal[
    "Ascon-Hash256",
    "Ascon-XOF128",
]

AsconCxofVariant: TypeAlias = Literal[
    "Ascon-CXOF128",
]

AsconMacVariant: TypeAlias = Literal[
    "Ascon-Mac",
    "Ascon-Prf",
    "Ascon-PrfShort"
]

AsconVariant: TypeAlias = AsconAeadVariant|AsconHashVariant|AsconCxofVariant|AsconMacVariant

debug = False
debugpermutation = False

# === Ascon hash/xof ===

def ascon_hash(message: BytesLike, variant: AsconHashVariant|AsconCxofVariant = "Ascon-Hash256", hashlength: int = 32, customization: BytesLike = b"") -> bytes:
    """
    Ascon hash function and extendable-output function.
    message: a bytes object of arbitrary length
    variant: "Ascon-Hash256" (with 256-bit output for 128-bit security), "Ascon-XOF128", or "Ascon-CXOF128" (both with arbitrary output length, security=min(128, bitlen/2))
    hashlength: the requested output bytelength (must be 32 for variant "Ascon-Hash256"; can be arbitrary for Ascon-XOF128, but should be >= 32 for 128-bit security)
    customization: a bytes object of at most 256 bytes specifying the customization string (only for Ascon-CXOF128)
    returns a bytes object containing the hash tag
    """
    versions = {"Ascon-Hash256": 2,
                "Ascon-XOF128": 3,
                "Ascon-CXOF128": 4}
    assert variant in versions.keys()
    if variant == "Ascon-Hash256": assert hashlength == 32
    if variant == "Ascon-CXOF128": assert len(customization) <= 256
    else: assert len(customization) == 0
    a = b = 12 # rounds
    rate = 8 # bytes
    taglen = 256 if variant == "Ascon-Hash256" else 0
    customize = True if variant == "Ascon-CXOF128" else False

    # Initialization
    iv = to_bytes([versions[variant], 0, (b<<4) + a]) + int_to_bytes(taglen, 2) + to_bytes([rate, 0, 0])
    S = bytes_to_state(iv + zero_bytes(32))
    if debug: printstate(S, "initial value:")

    ascon_permutation(S, 12)
    if debug: printstate(S, "initialization:")

    # Customization
    if customize:
        z_padding = to_bytes([0x01]) + zero_bytes(rate - (len(customization) % rate) - 1)
        z_length = int_to_bytes(len(customization)*8, 8)
        z_padded = z_length + customization + z_padding

        # customization blocks 0,...,m
        for block in range(0, len(z_padded), rate):
            S[0] ^= bytes_to_int(z_padded[block:block+rate])
            ascon_permutation(S, 12)
        if debug: printstate(S, "customization:")

    # Message Processing (Absorbing)
    m_padding = to_bytes([0x01]) + zero_bytes(rate - (len(message) % rate) - 1)
    m_padded = to_bytes(message) + to_bytes(m_padding)

    # message blocks 0,...,n
    for block in range(0, len(m_padded), rate):
        S[0] ^= bytes_to_int(m_padded[block:block+rate])
        ascon_permutation(S, 12)
    if debug: printstate(S, "process message:")

    # Finalization (Squeezing)
    H = b""
    while len(H) < hashlength:
        H += int_to_bytes(S[0], rate)
        ascon_permutation(S, 12)
    if debug: printstate(S, "finalization:")
    return H[:hashlength]


# === Ascon MAC/PRF ===

def ascon_mac(key: BytesLike, message: BytesLike, variant: AsconMacVariant = "Ascon-Mac", taglength: int = 16) -> bytes:
    """
    Ascon message authentication code (MAC) and pseudorandom function (PRF).
    key: a bytes object of size 16
    message: a bytes object of arbitrary length (<= 16 for "Ascon-PrfShort")
    variant: "Ascon-Mac" (128-bit output, arbitrarily long input), "Ascon-Prf" (arbitrarily long input and output), or "Ascon-PrfShort" (t-bit output for t<=128, m-bit input for m<=128)
    taglength: the requested output bytelength l/8 (must be <=16 for variants "Ascon-Mac" and "Ascon-PrfShort", arbitrary for "Ascon-Prf"; should be >= 16 for 128-bit security)
    returns a bytes object containing the authentication tag
    """
    assert variant in ("Ascon-Mac", "Ascon-Prf", "Ascon-PrfShort")
    if variant == "Ascon-Mac": assert len(key) == 16 and taglength <= 16
    if variant == "Ascon-Prf": assert len(key) == 16
    if variant == "Ascon-PrfShort": assert len(key) == 16 and taglength <= 16 and len(message) <= 16
    a = b = 12  # rounds
    msgblocksize = 32 # bytes (input rate for Mac, Prf)
    rate = 16 # bytes (output rate)

    # TODO update IVs to be consistent with NIST format

    if variant == "Ascon-PrfShort":
        # Initialization + Message Processing (Absorbing)
        IV = to_bytes([len(key) * 8, len(message)*8, a + 64, taglength * 8]) + zero_bytes(4)
        S = bytes_to_state(IV + key + message + zero_bytes(16 - len(message)))
        if debug: printstate(S, "initial value:")

        ascon_permutation(S, a)
        if debug: printstate(S, "process message:")

        # Finalization (Squeezing)
        T = int_to_bytes(S[3] ^ bytes_to_int(key[0:8]), 8) + int_to_bytes(S[4] ^ bytes_to_int(key[8:16]), 8)
        return T[:taglength]

    else: # Ascon-Prf, Ascon-Mac
        # Initialization
        if variant == "Ascon-Mac": tagspec = int_to_bytes(16*8, 4)
        elif variant == "Ascon-Prf": tagspec = int_to_bytes(0*8, 4)
        else: assert False, f"unknown variant {variant!r}"
        S = bytes_to_state(to_bytes([len(key) * 8, rate * 8, a + 128, a-b]) + tagspec + key + zero_bytes(16))
        if debug: printstate(S, "initial value:")

        ascon_permutation(S, a)
        if debug: printstate(S, "initialization:")

        # Message Processing (Absorbing)
        m_padding = to_bytes([0x01]) + zero_bytes(msgblocksize - (len(message) % msgblocksize) - 1)
        m_padded = to_bytes(message) + to_bytes(m_padding)

        # first s-1 blocks
        for block in range(0, len(m_padded) - msgblocksize, msgblocksize):
            S[0] ^= bytes_to_int(m_padded[block:block+8])     # msgblocksize=32 bytes
            S[1] ^= bytes_to_int(m_padded[block+8:block+16])
            S[2] ^= bytes_to_int(m_padded[block+16:block+24])
            S[3] ^= bytes_to_int(m_padded[block+24:block+32])
            ascon_permutation(S, b)
        # last block
        block = len(m_padded) - msgblocksize
        S[0] ^= bytes_to_int(m_padded[block:block+8])     # msgblocksize=32 bytes
        S[1] ^= bytes_to_int(m_padded[block+8:block+16])
        S[2] ^= bytes_to_int(m_padded[block+16:block+24])
        S[3] ^= bytes_to_int(m_padded[block+24:block+32])
        S[4] ^= 1
        if debug: printstate(S, "process message:")

        # Finalization (Squeezing)
        T = b""
        ascon_permutation(S, a)
        while len(T) < taglength:
            T += int_to_bytes(S[0], 8)  # rate=16
            T += int_to_bytes(S[1], 8)
            ascon_permutation(S, b)
        if debug: printstate(S, "finalization:")
        return T[:taglength]


# === Ascon AEAD encryption and decryption ===

def ascon_encrypt(key: BytesLike, nonce: BytesLike, associateddata: BytesLike, plaintext: BytesLike, variant: AsconAeadVariant = "Ascon-AEAD128") -> bytes:
    """
    Ascon encryption.
    key: a bytes object of size 16 (for Ascon-AEAD128; 128-bit security)
    nonce: a bytes object of size 16 (must not repeat for the same key!)
    associateddata: a bytes object of arbitrary length
    plaintext: a bytes object of arbitrary length
    variant: "Ascon-AEAD128"
    returns a bytes object of length len(plaintext)+16 containing the ciphertext and tag
    """
    versions = {"Ascon-AEAD128": 1}
    assert variant in versions.keys()
    assert len(key) == 16 and len(nonce) == 16
    S = [0, 0, 0, 0, 0]
    k = len(key) * 8   # bits
    a = 12   # rounds
    b = 8    # rounds
    rate = 16   # bytes

    ascon_initialize(S, k, rate, a, b, versions[variant], key, nonce)
    ascon_process_associated_data(S, b, rate, associateddata)
    ciphertext = ascon_process_plaintext(S, b, rate, plaintext)
    tag = ascon_finalize(S, rate, a, key)
    return ciphertext + tag


def ascon_decrypt(key: BytesLike, nonce: BytesLike, associateddata: BytesLike, ciphertext: BytesLike, variant: AsconAeadVariant = "Ascon-AEAD128") -> bytes|None:
    """
    Ascon decryption.
    key: a bytes object of size 16 (for Ascon-AEAD128; 128-bit security)
    nonce: a bytes object of size 16 (must not repeat for the same key!)
    associateddata: a bytes object of arbitrary length
    ciphertext: a bytes object of arbitrary length (also contains tag)
    variant: "Ascon-AEAD128"
    returns a bytes object containing the plaintext or None if verification fails
    """
    versions = {"Ascon-AEAD128": 1}
    assert variant in versions.keys()
    assert len(key) == 16 and len(nonce) == 16 and len(ciphertext) >= 16
    S = [0, 0, 0, 0, 0]
    k = len(key) * 8 # bits
    a = 12  # rounds
    b = 8   # rounds
    rate = 16   # bytes

    ascon_initialize(S, k, rate, a, b, versions[variant], key, nonce)
    ascon_process_associated_data(S, b, rate, associateddata)
    plaintext = ascon_process_ciphertext(S, b, rate, ciphertext[:-16])
    tag = ascon_finalize(S, rate, a, key)
    if tag == ciphertext[-16:]:
        return plaintext
    else:
        return None


# === Ascon AEAD building blocks ===

def ascon_initialize(S: list[int], k: int, rate: int, a: int, b: int, version: int, key: BytesLike, nonce: BytesLike):
    """
    Ascon initialization phase - internal helper function.
    S: Ascon state, a list of 5 64-bit integers
    k: key size in bits
    rate: block size in bytes (16 for Ascon-AEAD128)
    a: number of initialization/finalization rounds for permutation
    b: number of intermediate rounds for permutation
    version: 1 (for Ascon-AEAD128)
    key: a bytes object of size 16 (for Ascon-AEAD128; 128-bit security)
    nonce: a bytes object of size 16
    returns nothing, updates S
    """
    taglen = 128
    iv = to_bytes([version, 0, (b<<4) + a]) + int_to_bytes(taglen, 2) + to_bytes([rate, 0, 0])
    S[0], S[1], S[2], S[3], S[4] = bytes_to_state(iv + key + nonce)
    if debug: printstate(S, "initial value:")

    ascon_permutation(S, a)

    zero_key = bytes_to_state(zero_bytes(40-len(key)) + key)
    S[0] ^= zero_key[0]
    S[1] ^= zero_key[1]
    S[2] ^= zero_key[2]
    S[3] ^= zero_key[3]
    S[4] ^= zero_key[4]
    if debug: printstate(S, "initialization:")


def ascon_process_associated_data(S: list[int], b: int, rate: int, associateddata: BytesLike):
    """
    Ascon associated data processing phase - internal helper function.
    S: Ascon state, a list of 5 64-bit integers
    b: number of intermediate rounds for permutation
    rate: block size in bytes (16 for Ascon-AEAD128)
    associateddata: a bytes object of arbitrary length
    returns nothing, updates S
    """
    if len(associateddata) > 0:
        a_padding = to_bytes([0x01]) + zero_bytes(rate - (len(associateddata) % rate) - 1)
        a_padded = to_bytes(associateddata) + a_padding

        for block in range(0, len(a_padded), rate):
            S[0] ^= bytes_to_int(a_padded[block:block+8])
            if rate == 16:
                S[1] ^= bytes_to_int(a_padded[block+8:block+16])

            ascon_permutation(S, b)

    S[4] ^= 1<<63
    if debug: printstate(S, "process associated data:")


def ascon_process_plaintext(S: list[int], b: int, rate: int, plaintext: BytesLike):
    """
    Ascon plaintext processing phase (during encryption) - internal helper function.
    S: Ascon state, a list of 5 64-bit integers
    b: number of intermediate rounds for permutation
    rate: block size in bytes (16 for Ascon-AEAD128)
    plaintext: a bytes object of arbitrary length
    returns the ciphertext (without tag), updates S
    """
    p_lastlen = len(plaintext) % rate
    p_padding = to_bytes([0x01]) + zero_bytes(rate-p_lastlen-1)
    p_padded = to_bytes(plaintext) + p_padding

    # first t-1 blocks
    ciphertext = to_bytes([])
    for block in range(0, len(p_padded) - rate, rate):
        S[0] ^= bytes_to_int(p_padded[block:block+8])
        S[1] ^= bytes_to_int(p_padded[block+8:block+16])
        ciphertext += (int_to_bytes(S[0], 8) + int_to_bytes(S[1], 8))
        ascon_permutation(S, b)

    # last block t
    block = len(p_padded) - rate
    S[0] ^= bytes_to_int(p_padded[block:block+8])
    S[1] ^= bytes_to_int(p_padded[block+8:block+16])
    ciphertext += (int_to_bytes(S[0], 8)[:min(8,p_lastlen)] + int_to_bytes(S[1], 8)[:max(0,p_lastlen-8)])
    if debug: printstate(S, "process plaintext:")
    return ciphertext


def ascon_process_ciphertext(S: list[int], b: int, rate: int, ciphertext: BytesLike):
    """
    Ascon ciphertext processing phase (during decryption) - internal helper function.
    S: Ascon state, a list of 5 64-bit integers
    b: number of intermediate rounds for permutation
    rate: block size in bytes (16 for Ascon-AEAD128)
    ciphertext: a bytes object of arbitrary length
    returns the plaintext, updates S
    """
    c_lastlen = len(ciphertext) % rate
    c_padded = to_bytes(ciphertext) + zero_bytes(rate - c_lastlen)

    # first t-1 blocks
    plaintext = to_bytes([])
    for block in range(0, len(c_padded) - rate, rate):
        Ci = (bytes_to_int(c_padded[block:block+8]), bytes_to_int(c_padded[block+8:block+16]))
        plaintext += (int_to_bytes(S[0] ^ Ci[0], 8) + int_to_bytes(S[1] ^ Ci[1], 8))
        S[0] = Ci[0]
        S[1] = Ci[1]
        ascon_permutation(S, b)

    # last block t
    block = len(c_padded) - rate
    c_padx = zero_bytes(c_lastlen) + to_bytes([0x01]) + zero_bytes(rate-c_lastlen-1)
    c_mask = zero_bytes(c_lastlen) + ff_bytes(rate-c_lastlen)
    Ci = (bytes_to_int(c_padded[block:block+8]), bytes_to_int(c_padded[block+8:block+16]))
    plaintext += (int_to_bytes(S[0] ^ Ci[0], 8) + int_to_bytes(S[1] ^ Ci[1], 8))[:c_lastlen]
    S[0] = (S[0] & bytes_to_int(c_mask[0:8]))  ^ Ci[0] ^ bytes_to_int(c_padx[0:8])
    S[1] = (S[1] & bytes_to_int(c_mask[8:16])) ^ Ci[1] ^ bytes_to_int(c_padx[8:16])
    if debug: printstate(S, "process ciphertext:")
    return plaintext


def ascon_finalize(S: list[int], rate: int, a: int, key: BytesLike):
    """
    Ascon finalization phase - internal helper function.
    S: Ascon state, a list of 5 64-bit integers
    rate: block size in bytes (16 for Ascon-AEAD128)
    a: number of initialization/finalization rounds for permutation
    key: a bytes object of size 16 (for Ascon-AEAD128; 128-bit security)
    returns the tag, updates S
    """
    assert len(key) == 16
    S[rate//8+0] ^= bytes_to_int(key[0:8])
    S[rate//8+1] ^= bytes_to_int(key[8:16])

    ascon_permutation(S, a)

    S[3] ^= bytes_to_int(key[-16:-8])
    S[4] ^= bytes_to_int(key[-8:])
    tag = int_to_bytes(S[3], 8) + int_to_bytes(S[4], 8)
    if debug: printstate(S, "finalization:")
    return tag


# === Ascon permutation ===

def ascon_permutation(S: list[int], rounds: int=1):
    """
    Ascon core permutation for the sponge construction - internal helper function.
    S: Ascon state, a list of 5 64-bit integers
    rounds: number of rounds to perform
    returns nothing, updates S
    """
    assert rounds <= 12
    if debugpermutation: printwords(S, "permutation input:")
    for r in range(12-rounds, 12):
        # --- add round constants ---
        S[2] ^= (0xf0 - r*0x10 + r*0x1)
        if debugpermutation: printwords(S, "round constant addition:")
        # --- substitution layer ---
        S[0] ^= S[4]
        S[4] ^= S[3]
        S[2] ^= S[1]
        T = [(S[i] ^ 0xFFFFFFFFFFFFFFFF) & S[(i+1)%5] for i in range(5)]
        for i in range(5):
            S[i] ^= T[(i+1)%5]
        S[1] ^= S[0]
        S[0] ^= S[4]
        S[3] ^= S[2]
        S[2] ^= 0XFFFFFFFFFFFFFFFF
        if debugpermutation: printwords(S, "substitution layer:")
        # --- linear diffusion layer ---
        S[0] ^= rotr(S[0], 19) ^ rotr(S[0], 28)
        S[1] ^= rotr(S[1], 61) ^ rotr(S[1], 39)
        S[2] ^= rotr(S[2],  1) ^ rotr(S[2],  6)
        S[3] ^= rotr(S[3], 10) ^ rotr(S[3], 17)
        S[4] ^= rotr(S[4],  7) ^ rotr(S[4], 41)
        if debugpermutation: printwords(S, "linear diffusion layer:")


# === helper functions ===

def get_random_bytes(num: int) -> bytes:
    import os
    return to_bytes(os.urandom(num))

def zero_bytes(n: int) -> bytes:
    return n * b"\x00"

def ff_bytes(n: int) -> bytes:
    return n * b"\xFF"

def to_bytes(l: BytesLike|Iterable[int]) -> bytes:
    # where l is a list or bytearray or bytes
    return bytes(l)

def bytes_to_int(bytes: BytesLike) -> int:
    return int.from_bytes(bytes, 'little')
    return sum([bi << (i*8) for i, bi in enumerate(to_bytes(bytes))])

def bytes_to_state(bytes: bytes) -> list[int]:
    return [bytes_to_int(bytes[8*w:8*(w+1)]) for w in range(5)]

def int_to_bytes(integer: int, nbytes: int) -> bytes:
    return integer.to_bytes(nbytes, 'little')

def rotr(val: int, r: int) -> int:
    return (val >> r) | ((val & (1<<r)-1) << (64-r))

def bytes_to_hex(b: bytes) -> str:
    return b.hex()

def printstate(S: list[int], description: str = "") -> None:
    print(" " + description)
    print(" ".join(["{s:016x}".format(s=s) for s in S]))

def printwords(S: list[int], description: str = "") -> None:
    print(" " + description)
    print("\n".join(["  x{i}={s:016x}".format(**locals()) for i, s in enumerate(S)]))


# === some demo if called directly ===

def demo_print(data: list[tuple[str, bytes|None]]) -> None:
    maxlen = max([len(text) for (text, val) in data])
    for text, val in data:
        val_ = bytes_to_hex(val) if val is not None else None
        print("{text}:{align} 0x{val} ({length} bytes)".format(text=text, align=((maxlen - len(text)) * " "), val=val_, length=len(val_) if val_ is not None else 0))

def demo_aead(variant: AsconAeadVariant = "Ascon-AEAD128") -> None:
    assert variant in ("Ascon-AEAD128")
    print("=== demo encryption using {variant} ===".format(variant=variant))

    # choose a cryptographically strong random key and a nonce that never repeats for the same key:
    key   = b'0123456789ABCDEF'  # zero_bytes(16)
    nonce = b'ABCDEF0123456789'  # zero_bytes(16)

    associateddata = b""
    plaintext      = b"Hello World"

    ciphertext        = ascon_encrypt(key, nonce, associateddata, plaintext,  variant)
    receivedplaintext = ascon_decrypt(key, nonce, associateddata, ciphertext, variant)

    if receivedplaintext is None: print("verification failed!")

    demo_print([("key", key),
                ("nonce", nonce),
                ("plaintext", plaintext),
                ("ass.data", associateddata),
                ("ciphertext", ciphertext[:-16]),
                ("tag", ciphertext[-16:]),
                ("received", receivedplaintext),
               ])

def demo_hash(variant: AsconHashVariant|AsconCxofVariant = "Ascon-Hash256", hashlength: int = 32) -> None:
    assert variant in ("Ascon-Hash256", "Ascon-XOF128", "Ascon-CXOF128")
    print("=== demo hash using {variant} ===".format(variant=variant))

    message = b"ascon"
    customization = b"custom" if variant == "Ascon-CXOF128" else b""
    tag = ascon_hash(message, variant, hashlength, customization)

    demo_print([("message", message), ("customization", customization), ("tag", tag)])

def demo_mac(variant: AsconMacVariant = "Ascon-Mac", taglength: int = 16) -> None:
    # TODO rename variants to be consistent with NIST format
    assert variant in ("Ascon-Mac", "Ascon-Prf", "Ascon-PrfShort")
    print("=== demo MAC using {variant} ===".format(variant=variant))

    key = get_random_bytes(16)
    message = b"ascon"
    tag = ascon_mac(key, message, variant)

    demo_print([("key", key), ("message", message), ("tag", tag)])


if __name__ == "__main__":
    demo_aead("Ascon-AEAD128")
    demo_hash("Ascon-Hash256")
    demo_hash("Ascon-XOF128")
    demo_hash("Ascon-CXOF128")
    demo_mac("Ascon-Mac")

=== demo encryption using Ascon-AEAD128 ===
key:        0x30313233343536373839414243444546 (32 bytes)
nonce:      0x41424344454630313233343536373839 (32 bytes)
plaintext:  0x48656c6c6f20576f726c64 (22 bytes)
ass.data:   0x (0 bytes)
ciphertext: 0x368f11357761d55d0dcfab (22 bytes)
tag:        0x2cc86521a9446ba1acc3b9dd0b4b50f9 (32 bytes)
received:   0x48656c6c6f20576f726c64 (22 bytes)
=== demo hash using Ascon-Hash256 ===
message:       0x6173636f6e (10 bytes)
customization: 0x (0 bytes)
tag:           0x65904928ac016bc02577b23b3f79e336fdf43b6d81746058979c6cd67630a593 (64 bytes)
=== demo hash using Ascon-XOF128 ===
message:       0x6173636f6e (10 bytes)
customization: 0x (0 bytes)
tag:           0x7c46dac982bc03ecc63cc7af25485013486eabab61b8a963467245770d6e0aab (64 bytes)
=== demo hash using Ascon-CXOF128 ===
message:       0x6173636f6e (10 bytes)
customization: 0x637573746f6d (12 bytes)
tag:           0x4133c59dc1b1957933d12c759fc442b744795c8cde075f5607d1a6668f124959 (64 bytes)
=== dem